In [1]:
#IMPORTS
import pandas as pd
from datetime import datetime
import time
from time import sleep
import re

from agents import *
from papers import *
import os
from datetime import datetime
import json

/opt/homebrew/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/opt/homebrew/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# get accession
df=pd.read_csv('data/example/example_data.csv')
accessions=df['Accession']

In [3]:
#Create library


# Create csv file
with open('results/library/library_summary.csv', 'w') as csvfile1:
    csvfile1.write(f'accession,number_of_papers,pmc_papers,pm_papers,duration(s)\n')

# Create txt file
with open('results/library/library_curation.txt', 'w') as infile:
    infile.write(f'Summary of the library Curation for example data\n{datetime.now()}\n\n')

# create library for all accessions
full_library_start=time.time()
for accession in accessions:
    # get start time
    start=time.time()
    # Create library
    CreateLibrary(accession)
    #End time
    end=time.time()
    #Duration
    duration=end-start

    #Number of papers
    papers=sorted(os.listdir(f'data/library/{accession}'))

    # Number of PMC papers
    pmc_papers=0
    for paper in papers:
        if paper.startswith('PMC'):
            pmc_papers+=1

    pm_papers=(len(papers)-pmc_papers)-1 # -1 for json

    with open(f'results/library/library_curation.txt','a') as f:
        f.write(f'\n\n\n================{accession}===============\n')
        f.write(f'Number of papers: {len(papers)}\n')
        f.write(f'Number of PMCs: {pmc_papers}\n')
        f.write(f'Number of PM papers: {pm_papers}\n')
        f.write(f'Duration: {round(duration,2)} seconds\n')

        f.write(f'======Paper Summary======\n')
        for paper in papers:
            with open(f'data/library/{accession}/{paper}','r') as infile:
                length=len(infile.read())
            f.write(f'{paper}\nNumber of characters: {length}\n')

    with open(f'results/library/library_summary.csv', 'a') as csvfile:
        csvfile.write(f'{accession},{len(papers)},{pmc_papers},{pm_papers},{duration}\n')




full_library_end=time.time()
full_duration=(full_library_end-full_library_start)/60
with open('results/library/library_curation.txt','a') as f:
    f.write(f'\n\nTotal Duration (minutes): {round(full_duration,2)}\n')

Tequatrovirus T4


KeyboardInterrupt: 

In [ ]:
# Identify Bacterial Host from metadata

with open('results/host/host_classification.csv', 'w') as csvfile1:
    csvfile1.write(f'accession,host,taxonomic_level,reasoning,source,duration(m)\n')

with open('results/host/host_metadata_classification.txt','w') as f:
    f.write(f'Host classification of example data using metadata\n{datetime.now()}\n\n')

full_Start=time.time()
for accession in accessions:
    print(f'=============={accession}=============')
    with open(f"data/library/{accession}/{accession}.json", "r") as inj:
        metadata = json.load(inj)
        print(f'Metadata: {metadata}')

        # Define the start
    state = {
        'paper_text': None,
        'source_path': None,
        'model': 'qwen3.5',
        'metadata': metadata, # only including metadata
        'phage': None,
        'host_species': None,
        'reasoning': None,
        'confidence': None,
        'thermal_range': None,
        'temperature': None,
        'thermal_reasoning': None,
        'thermal_confidence': None,
    }

    start = time.time()
    output=HostMetadata(state)
    end=time.time()
    duration=(end-start)/60
    print(f'Accession: {accession}\nHost Species: {output.get("host_species", "unknown")}')
    print(f'Duration (minutes): {round(duration,2)}')
    print(f'LLM Output: {output}\n\n')

    if output.get('host_species', 'unknown') != 'unknown':
        source='metadata'
    else:
        source='unknown'


    reasoning=output.get('host_reasoning', 'unknown')
    reasoning=reasoning.replace(',', '')

    with open(f'results/host/host_classification.csv', 'a') as csvfile1:
        csvfile1.write(f"{accession},{output.get('host_species','unknown')},{output.get('taxon_level','unknown')},{reasoning},{source},{round(duration,2)}\n")

    with open(f'results/host/host_metadata_classification.txt', 'a') as f:
        f.write(f'==============={accession}==============\n')
        f.write(f'Host Species: {output.get("host_species", "unknown")}\n')
        f.write(f'Taxonomic Level: {output.get("taxon_level", "unknown")}\n')
        f.write(f'Reasoning: {output.get("host_reasoning", "unknown")}\n')
        f.write(f'Duration: {round(duration,2)} minutes\n')


full_end=time.time()
full_duration=(full_end-full_Start)/60

with open('results/host/host_metadata_classification.txt', 'a') as outf:
    outf.write(f'\n\nTotal Duration (minutes): {round(full_duration,2)}\n')


In [ ]:
# Identify thermal range from metadata
with open('results/thermal/thermal_metadata_classification.txt', 'w') as outf:
     outf.write(f'Thermal classification of example data using accession metadata\n{datetime.now()}\n\n')

with open('results/thermal/thermal_classification.csv', 'w') as csvout:
    csvout.write('accession,thermal_range,inference_type,reasoning,confidence,source,duration(m)\n')

full_start=time.time()
for accession in accessions:
    print(f'=============={accession}=============')
    with open(f"data/library/{accession}/{accession}.json", "r") as f:
        metadata = json.load(f)
        print(f'Metadata: {metadata}')

        # Define the start
    state = {
        'paper_text': None,
        'source_path': None,
        'model': 'qwen3.5',
        'metadata': metadata, # only including metadata
        'phage': None,
        'host_species': None,
        'reasoning': None,
        'confidence': None,
        'thermal_range': None,
        'temperature': None,
        'thermal_reasoning': None,
        'thermal_confidence': None,
    }

    start = time.time()
    output=ThermalMetadata(state)
    end=time.time()
    duration=(end-start)/60
    print(f'\nThermal Range: {output.get("thermal_range", "unknown")}')
    print(f'Duration (minutes): {round(duration,2)}')
    print(f'LLM Output: {output}\n\n')

    if output.get('thermal_range', 'unknown') != 'unknown':
        source='metadata'
    else:
        source='unknown'

    reasoning=output.get("reasoning", "unknown")
    reasoning=reasoning.replace(',', '')

    with open(f'results/thermal/thermal_metadata_classification.txt', 'a') as f:
        f.write(f'==============={accession}==============\n')
        f.write(f'Thermal Range {output.get("thermal_range", "unknown")}\n')
        f.write(f'Inference Type {output.get("inference_type", "unknown")}\n')
        f.write(f'Reasoning {output.get("reasoning", "unknown")}\n')
        f.write(f'Confidence {output.get("confidence", "unknown")}\n')
        f.write(f'Duration: {round(duration,2)} minutes\n')


    with open(f'results/thermal/thermal_classification.csv', 'a') as f:
        f.write(f"{accession},{output.get('thermal_range', 'unknown')},{output.get('inference_type', 'unknown')},{reasoning},{output.get('confidence', 'unknown')},{source},{round(duration,2)}\n")

full_end=time.time()
full_duration=(full_end-full_start)/60

with open('results/thermal/thermal_metadata_classification.txt', 'a') as f:
    f.write(f'\n\nTotal Duration (minutes): {round(full_duration,2)}\n')

In [4]:
# CLASSIFYING HOST ORGANISM USING LITERATURE


patterns = {
    "infection_terms": r"\b(infects?|infecting|infected)\b",
    "isolation_terms": r"\b(isolated from|recovered from|obtained from|detected in)\b",
    "host_terms": r"\b(host bacterium|host bacteria|bacterial host)\b",
    "phage_of": r"\bphage (?:of|infecting)\b",
}

df=pd.read_csv('results/host/host_classification.csv')
accessions=df[df['host']=='unknown']['accession']

with open(f'results/host/host_literature_classification.txt', 'w') as outfile:
    outfile.write('Host Classification of Example data using first literature creation\n')
    outfile.write(f'Attempts to classify the accessions not classified from metadata search\n')
    outfile.write(f'Uses the first literature creation pulled from the protein accession\n')
    outfile.write(f'{datetime.now()}\n')


full_start=time.time()
for accession in accessions:
    acc_start=time.time()
    with open(f'results/host/host_literature_classification.txt', 'a') as f:
        f.write(f'======================{accession}======================\n')

    rankings=RankPapers(f'data/library/{accession}/', patterns)

    prev_duration=df[df['accession']==accession]['duration(m)'].values[0]

    with open(f'data/library/{accession}/{accession}.json', 'r') as f:
        metadata = json.load(f)

    papers=list(rankings.keys())
    host_high_confidence = False

    for paper in papers:
        with open(f'data/library/{accession}/{paper}') as f:
            paper_text = f.read()

        state = {
            'paper_text': paper_text,
            'source_path': f'data/library/{accession}/{paper}',
            'model': 'qwen3.5',
            'metadata': metadata,
            'phage': metadata.get('organism', 'unknown'),
            'host_species': None,
            'reasoning': None,
            'confidence': None,
            'thermal_range': None,
            'temperature': None,
            'thermal_reasoning': None,
            'thermal_confidence': None,
        }

        start = time.time()
        host_classification = IdentifyHost(state)
        print(f'LLM output After JSONIFICATION: {host_classification}')
        end = time.time()
        duration=(end-start)/60


        host=host_classification.get('host_species','unknown')
        reasoning=host_classification.get('host_reasoning','unknown')
        confidence=host_classification.get('host_confidence','unknown')

        reasoning=reasoning.replace(',', '')


        with open(f'results/host/host_literature_classification.txt', 'a') as f:
            f.write(f'===================\nPaper: {paper}\tPaper Score: {rankings[paper]["total"]}\n')
            f.write(f'Host Species: {host}\n')
            f.write(f'Reasoning: {reasoning}\n')
            f.write(f'Confidence: {confidence}\n')
            f.write(f'Duration: {round(duration,2)}\n')

        if host != 'unknown' and confidence == 'high':
            host_high_confidence = True
            df.loc[df['accession'] == accession, 'host'] = host
            df.loc[df['accession'] == accession, 'duration(m)'] = prev_duration + duration
            df.loc[df['accession'] == accession, 'source'] = 'literature'
            df.loc[df['accession'] == accession, 'reasoning'] = reasoning
            break


    if not host_high_confidence:
        df.loc[df['accession'] == accession, 'host'] = 'unknown'


    acc_end=time.time()
    acc_duration=(acc_end-acc_start)/60
    with open(f'results/host/host_literature_classification.txt', 'a') as f:
        f.write(f'\nFull Accession Duration: {round(acc_duration,2)}minutes\n')

full_end=time.time()
full_duration=(full_end-full_start)/60
with open('results/host/host_literature_classification.txt', 'a') as f:
    f.write(f'\n\nTotal Duration (minutes): {round(full_duration,2)}')

df.to_csv('results/host/host_classification.csv')



LLM OUTPUT: content='' additional_kwargs={} response_metadata={'model': 'qwen3.5', 'created_at': '2026-04-29T15:26:48.61998Z', 'done': True, 'done_reason': 'length', 'total_duration': 272292731542, 'load_duration': 93844542, 'prompt_eval_count': 28512, 'prompt_eval_duration': 88417719250, 'eval_count': 4096, 'eval_duration': 182261168541, 'logprobs': None, 'model_name': 'qwen3.5', 'model_provider': 'ollama'} id='lc_run--019dd9d5-6d85-7cd3-8821-22dc41c8c4be-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 28512, 'output_tokens': 4096, 'total_tokens': 32608}
LLM output After JSONIFICATION: {}
LLM OUTPUT: content='' additional_kwargs={} response_metadata={'model': 'qwen3.5', 'created_at': '2026-04-29T15:30:25.509566Z', 'done': True, 'done_reason': 'length', 'total_duration': 216870095250, 'load_duration': 78829959, 'prompt_eval_count': 16367, 'prompt_eval_duration': 47767781667, 'eval_count': 4096, 'eval_duration': 167534579266, 'logprobs': None, 'model_name': 'qwen3

In [5]:
# Classify Thermal Range using initial literature search
from papers import *
from datetime import datetime
import pandas as pd
import time
from agents import IdentifyThermalRange

patterns = {
    # organisms
    "mesophile": r"\bmesophil(?:e|ic|es|icity)?\b",
    "psychrophile": r"\bpsychrophil(?:e|ic|es|icity)?\b",
    "thermophile": r"\bthermophil(?:e|ic|es|icity)?\b",
    # temperature concepts
    "temperature": r"\btemperatur(e|es)?\b",
    "optimal_growth": r"\boptimal(?:ly)?\s+(growth|temperature|growing)\b",
    # units
    "celsius": r"\b\d+(?:\.\d+)?\s?°?\s?[Cc]\b|\bCelsius\b",
    "kelvin": r"\b\d+(?:\.\d+)?\s?[Kk]\b|\bKelvin\b"
}

df=pd.read_csv('results/thermal/thermal_classification.csv')
accessions=df[df['thermal_range']=='unknown']['accession']

host_df=pd.read_csv('results/host/host_classification.csv')

with open(f'results/thermal/thermal_literature_classification.txt', 'w') as outfile:
    outfile.write('Thermal Classification of Example data using first literature creation\n')
    outfile.write(f'Attempts to classify the accessions not classified from metadata search\n')
    outfile.write(f'Uses the first literature creation pulled from the protein accession, not literature of the host\n')
    outfile.write(f'{datetime.now()}\n')

full_Start=time.time()
for accession in accessions:
    acc_start=time.time()
    with open(f'results/thermal/thermal_literature_classification.txt', 'a') as f:
        f.write(f'======================{accession}======================\n')

    rankings=RankPapers(f'data/library/{accession}/', patterns)

    host=host_df[host_df['accession']==accession]['host'].values[0]

    prev_duration=df[df['accession']==accession]['duration(m)'].values[0]


    with open(f'data/library/{accession}/{accession}.json', 'r') as f:
        metadata = json.load(f)

    papers=list(rankings.keys())
    thermal_high_confidence = False

    for paper in papers:
        with open(f'data/library/{accession}/{paper}') as f:
            paper_text = f.read()


        state = {
            'paper_text': paper_text,
            'source_path': f'data/library/{accession}/{paper}',
            'model': 'qwen3.5',
            'metadata': metadata,
            'phage': metadata.get('organism', 'unknown'),
            'host_species': host,
            'reasoning': None,
            'confidence': None,
            'thermal_range': None,
            'temperature': None,
            'thermal_reasoning': None,
            'thermal_confidence': None,
        }

        start = time.time()
        thermal_classification = IdentifyThermalRange(state)
        print(f'LLM output: {thermal_classification}')
        end = time.time()
        duration=(end-start)/60

        thermal_range=thermal_classification.get('thermal_range', 'unknown')
        temperature=thermal_classification.get('temperature', 'unknown')
        reasoning=thermal_classification.get('thermal_reasoning', 'unknown')
        confidence=thermal_classification.get('thermal_confidence', 'unknown')

        reasoning=reasoning.replace(',', '')

        with open(f'results/thermal/thermal_literature_classification.txt', 'a') as f:
            f.write(f'===================\nPaper: {paper}\tPaper Score: {rankings[paper]["total"]}\n')
            f.write(f'Thermal Range: {thermal_range}\n')
            f.write(f'Temperature: {temperature}\n')
            f.write(f'Reasoning: {reasoning}\n')
            f.write(f'Confidence: {confidence}\n')
            f.write(f'Duration (minutes): {round(duration,2)}\n\n')

        if thermal_range != 'unknown' and confidence == 'high':
            thermal_high_confidence = True
            df.loc[df['accession'] == accession, 'thermal_range'] = thermal_range
            df.loc[df['accession'] == accession, 'duration(m)'] = prev_duration + duration
            df.loc[df['accession'] == accession, 'source'] = 'literature'
            df.loc[df['accession'] == accession, 'reasoning'] = reasoning
            break

    if not thermal_high_confidence:
        df.loc[df['accession'] == accession, 'thermal_range'] = 'unknown'

    acc_end=time.time()
    acc_duration=(acc_end-acc_start)/60
    with open(f'results/thermal/thermal_literature_classification.txt', 'a') as f:
        f.write(f'\nFull Accession Duration: {round(acc_duration,2)}minutes\n')
full_end=time.time()
full_duration=(full_end-full_Start)/60
with open('results/thermal/thermal_literature_classification.txt', 'a') as f:
    f.write(f'\n\nTotal Duration (minutes): {round(full_duration,2)}')

df.to_csv('results/thermal/thermal_classification.csv')

LLM output: {}
LLM output: {}
LLM output: {}
LLM output: {}


KeyboardInterrupt: 

In [ ]:
df